# Struktur Dataset

## Install & Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Upload & Load Dataset

In [1]:
from google.colab import files
uploaded = files.upload()

import io
df = pd.read_csv(io.BytesIO(uploaded['shopping_trends_updated.csv']))

print(f"✅ Dataset berhasil dimuat!")
print(f"📊 Jumlah baris: {df.shape[0]}")
print(f"📊 Jumlah kolom: {df.shape[1]}")

Saving shopping_trends_updated.csv to shopping_trends_updated.csv


NameError: name 'pd' is not defined

## Melihat Struktur Data

In [ ]:
print("=" * 60)
print("📌 PREVIEW DATA (5 baris pertama):")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("📌 INFO DATASET:")
print("=" * 60)
df.info()

 ## Deskripsi Kolom & Statistik

In [ ]:
print("=" * 60)
print("📌 STATISTIK DESKRIPTIF (Kolom Numerik):")
print("=" * 60)
display(df.describe())


print("\n" + "=" * 60)
print("📌 DAFTAR KOLOM:")
print("=" * 60)
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

## Cek Missing Values & Duplikat

In [ ]:
print("=" * 60)
print("📌 MISSING VALUES PER KOLOM:")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("✅ Tidak ada missing values!")

print("\n" + "=" * 60)
print("📌 CEK DUPLIKASI:")
print("=" * 60)
duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {duplikat}")
if duplikat == 0:
    print("✅ Tidak ada data duplikat!")

##  Cek Unique Values Kolom Kategorikal

In [ ]:
print("=" * 60)
print("📌 UNIQUE VALUES — KOLOM KATEGORIKAL:")
print("=" * 60)

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    unique_vals = df[col].unique()
    print(f"\n🔹 {col} ({len(unique_vals)} nilai unik):")
    print(f"   {list(unique_vals)}")

# Data Cleaning & Preprocessing

## Rename Kolom (snake_case untuk PostgreSQL)

In [ ]:
df.columns = [col.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for col in df.columns]

print("✅ Kolom setelah rename:")
for col in df.columns:
    print(f"   → {col}")

## Konversi Tipe Data

In [ ]:
bool_cols = ['subscription_status', 'discount_applied', 'promo_code_used']
for col in bool_cols:
    df[col] = df[col].map({'Yes': True, 'No': False})

# Pastikan tipe numerik sudah benar
df['purchase_amount_usd'] = df['purchase_amount_usd'].astype(float)
df['review_rating'] = df['review_rating'].astype(float)

print("✅ Tipe data setelah konversi:")
print(df[bool_cols + ['purchase_amount_usd', 'review_rating']].dtypes)

## Tambahkan kolom age ke grup

In [ ]:
def categorize_age(age):
    if age < 18:
        return 'Teen'
    elif age < 26:
        return 'Young Adult'
    elif age < 40:
        return 'Adult'
    elif age < 60:
        return 'Middle Age'
    else:
        return 'Senior'

df['age_group'] = df['age'].apply(categorize_age)

print("✅ Distribusi Age Group:")
print(df['age_group'].value_counts())

## Tambah Kolom Frequency Score

In [ ]:
frequency_map = {
    'Weekly'        : 52,
    'Bi-Weekly'     : 26,
    'Fortnightly'   : 26,
    'Monthly'       : 12,
    'Quarterly'     : 4,
    'Every 3 Months': 4,
    'Annually'      : 1
}

df['frequency_score'] = df['frequency_of_purchases'].map(frequency_map)

print("✅ Frequency Score mapping:")
print(df[['frequency_of_purchases', 'frequency_score']].drop_duplicates().sort_values('frequency_score', ascending=False))

## Tambah Kolom Revenue Segment

In [ ]:
def revenue_segment(amount):
    if amount < 30:
        return 'Low'
    elif amount < 70:
        return 'Medium'
    else:
        return 'High'

df['revenue_segment'] = df['purchase_amount_usd'].apply(revenue_segment)

print("✅ Distribusi Revenue Segment:")
print(df['revenue_segment'].value_counts())

## Tambah Transaction ID & Verifikasi Final

In [ ]:
df.insert(0, 'transaction_id', range(1, len(df) + 1))

print("=" * 60)
print("📌 PREVIEW DATA FINAL:")
print("=" * 60)
display(df.head(3))

print(f"\n📊 Shape final: {df.shape}")
print(f"📋 Kolom final ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col} — {df[col].dtype}")

# File Bersih

In [ ]:
df.to_csv('shopping_trends_cleaned.csv', index=False)
print("✅ Data bersih disimpan sebagai 'shopping_trends_cleaned.csv'")

import os
print(os.getcwd())

# Proses ETL

## Tabel Dimensi & Fakta


In [ ]:
dim_customer = df[[
    'customer_id', 'age', 'gender', 'age_group', 'location',
    'subscription_status', 'previous_purchases',
    'frequency_of_purchases', 'frequency_score'
]].drop_duplicates(subset='customer_id').reset_index(drop=True)

dim_product = df[[
    'item_purchased', 'category', 'size', 'color'
]].drop_duplicates().reset_index(drop=True)
dim_product.insert(0, 'product_id', range(1, len(dim_product) + 1))

dim_season = df[['season']].drop_duplicates().reset_index(drop=True)
dim_season.insert(0, 'season_id', range(1, len(dim_season) + 1))

dim_shipping = df[[
    'shipping_type', 'discount_applied',
    'promo_code_used', 'payment_method'
]].drop_duplicates().reset_index(drop=True)
dim_shipping.insert(0, 'shipping_id', range(1, len(dim_shipping) + 1))

fact = df.merge(dim_product, on=['item_purchased','category','size','color'], how='left')
fact = fact.merge(dim_season, on='season', how='left')
fact = fact.merge(dim_shipping,
                  on=['shipping_type','discount_applied','promo_code_used','payment_method'],
                  how='left')

fact_sales = fact[[
    'transaction_id', 'customer_id', 'product_id',
    'season_id', 'shipping_id', 'purchase_amount_usd',
    'review_rating', 'revenue_segment'
]]

print("✅ Semua tabel berhasil disiapkan!")
print(f"   dim_customer : {dim_customer.shape}")
print(f"   dim_product  : {dim_product.shape}")
print(f"   dim_season   : {dim_season.shape}")
print(f"   dim_shipping : {dim_shipping.shape}")
print(f"   fact_sales   : {fact_sales.shape}")

## Fungsi Generate INSERT SQL

In [ ]:
def df_to_insert_sql(df, table_name):
    """Konversi DataFrame menjadi string SQL INSERT statements"""
    lines = []
    cols = ', '.join(df.columns)
    lines.append(f"-- INSERT INTO {table_name}")
    lines.append(f"-- Total rows: {len(df)}\n")

    for _, row in df.iterrows():
        values = []
        for val in row:
            if val is None or (isinstance(val, float) and pd.isna(val)):
                values.append('NULL')
            elif isinstance(val, bool):
                values.append('TRUE' if val else 'FALSE')
            elif isinstance(val, (int, float)):
                values.append(str(val))
            else:
                # Escape single quotes dalam string
                escaped = str(val).replace("'", "''")
                values.append(f"'{escaped}'")
        vals = ', '.join(values)
        lines.append(f"INSERT INTO {table_name} ({cols}) VALUES ({vals});")

    return '\n'.join(lines)

print("✅ Fungsi generate SQL siap!")

## Generate & Simpan File SQL

In [ ]:
print("🚀 Generating SQL files...")

sql_customer = df_to_insert_sql(dim_customer, 'dim_customer')
sql_product  = df_to_insert_sql(dim_product,  'dim_product')
sql_season   = df_to_insert_sql(dim_season,   'dim_season')
sql_shipping = df_to_insert_sql(dim_shipping, 'dim_shipping')
sql_fact     = df_to_insert_sql(fact_sales,   'fact_sales')

full_sql = f"""
-- ============================================================
-- DATA WAREHOUSE — SHOPPING TRENDS
-- ETL INSERT STATEMENTS
-- Generated dari Google Colab
-- ============================================================

-- Urutan insert: dimension dulu, baru fact table

-- ============================================================
{sql_customer}

-- ============================================================
{sql_product}

-- ============================================================
{sql_season}

-- ============================================================
{sql_shipping}

-- ============================================================
{sql_fact}
"""

with open('insert_data.sql', 'w', encoding='utf-8') as f:
    f.write(full_sql)

print("✅ File 'insert_data.sql' berhasil dibuat!")
print(f"📄 Estimasi jumlah baris SQL: {full_sql.count(chr(10))} baris")

## Download File SQL

In [ ]:
from google.colab import files

files.download('insert_data.sql')
print("✅ File sedang didownload ke laptop Anda!")